# 01_sync_and_normalize

## 目的
- 都道府県単位でデータを同期
- 正規化・Parquet 保存・DuckDB 保存
- 件数・重複確認

## 注意
全国同期は CLI: python sync_public_notice.py --year 2026 --z 13

In [1]:
import sys, os
from pathlib import Path

# プロジェクトルートを config.py の存在で探索（CWD に依存しない）
def _find_project_root() -> str:
    # Jupyter が提供する _dh（notebook ディレクトリ）を優先
    try:
        _nb_dir = Path(globals()["_dh"][0])
    except (KeyError, IndexError):
        _nb_dir = Path.cwd()
    for candidate in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]:
        if (candidate / "config.py").exists():
            return str(candidate.resolve())
    # フォールバック: 絶対パスで直接指定
    fallback = Path("/home/kazumasa/projects/land_price_api_app")
    if fallback.exists():
        return str(fallback)
    raise FileNotFoundError("プロジェクトルートが見つかりません")

_project_root = _find_project_root()
os.chdir(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)
print("作業ディレクトリ:", os.getcwd())


作業ディレクトリ: /home/kazumasa/projects/land_price_api_app


## 1. 環境確認

In [2]:
from notebook_utils import load_env_and_connect
conn = load_env_and_connect()

✓ APIキーを確認しました
✓ DB 接続成功: 38,227 レコード / 年度: [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018]


## 2. クリーンアップ（Parquet + DB + タイルキャッシュ削除）

既存データをすべて削除してから、主要7都府県 × 直近3年分を再取得します。

In [ ]:
# ⚠ 安全スイッチ: True に変更してから実行してください
CONFIRM_DELETE = False

if not CONFIRM_DELETE:
    print("❌ 実行をキャンセルしました。")
    print("   CONFIRM_DELETE = True に変更してから再実行してください。")
    raise SystemExit("削除確認スイッチがオフです")

from pathlib import Path
import db

processed_dir = Path("data/processed")
raw_dir = Path("data/raw")

# ── Parquet 削除 ────────────────────────────────────────────────
parquet_files = list(processed_dir.glob("*.parquet"))
for p in parquet_files:
    p.unlink()
print(f"Parquet 削除: {len(parquet_files)} ファイル")

# ── タイルキャッシュ削除 ─────────────────────────────────────────
cache_files = list(raw_dir.glob("fetched_tiles_*.json"))
for p in cache_files:
    p.unlink()
print(f"タイルキャッシュ削除: {len(cache_files)} ファイル")

# ── DB クリア ────────────────────────────────────────────────────
conn.execute("DELETE FROM land_prices_public_notice")
conn.commit()
stats = db.get_stats(conn)
print(f"✅ DB クリア完了: 残レコード {stats['total_records']} 件")

## 3. 主要7都府県 × 直近3年分 一括同期

`overwrite=True` でタイルキャッシュを無視して再取得します。  
目安: 約20〜40分（都府県 × 年 × API レート次第）

In [ ]:
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import sync_public_notice as sync
import db

# ── 設定 ─────────────────────────────────────────────────────────────────────
TARGET_PREFS = ["47", "13", "14", "27", "01", "23", "40"]
TARGET_YEARS = [2024, 2025, 2026]

PREF_NAMES = {
    "01": "北海道", "13": "東京都", "14": "神奈川県",
    "23": "愛知県", "27": "大阪府", "40": "福岡県", "47": "沖縄県",
}

TILE_WORKERS = 4   # 1都府県内のタイル並列取得数（目安: 4〜5）
PREF_WORKERS = 3   # 同時に処理する都府県数（目安: 2〜3）

processed_dir = Path("data/processed")

# ── Phase 1: 並列取得（Parquetのみ保存、DB書き込みはPhase2でまとめて） ─────────
_log_lock = threading.Lock()
results: dict[tuple, int] = {}

def _sync_one(pref: str, year: int) -> tuple:
    t0 = time.time()
    try:
        df = sync.sync_public_notice_year(
            year=year, z=13,
            overwrite=True,           # タイルキャッシュを無視して再取得
            prefecture_code=pref,
            conn=None,                # DB書き込みはPhase2で行う
            max_workers=TILE_WORKERS, # タイル並列取得
            skip_db_write=True,       # Parquetのみ保存
        )
        elapsed = time.time() - t0
        cnt = len(df) if df is not None and not df.empty else 0
        with _log_lock:
            mark = "✓" if cnt > 0 else "⚠"
            print(f"  {mark} {PREF_NAMES.get(pref, pref)} ({pref}) {year}年: {cnt:,} 件  {elapsed:.0f}秒")
        return pref, year, cnt, None
    except Exception as exc:
        elapsed = time.time() - t0
        with _log_lock:
            print(f"  ✗ {PREF_NAMES.get(pref, pref)} ({pref}) {year}年: エラー: {exc}  {elapsed:.0f}秒")
        return pref, year, 0, exc

total_start = time.time()
tasks = [(p, y) for p in TARGET_PREFS for y in TARGET_YEARS]
print(f"=== Phase 1: 並列取得 ===")
print(f"開始: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"対象: {len(TARGET_PREFS)} 都府県 × {len(TARGET_YEARS)} 年 = {len(tasks)} タスク")
print(f"並列設定: TILE_WORKERS={TILE_WORKERS}, PREF_WORKERS={PREF_WORKERS}\n")

with ThreadPoolExecutor(max_workers=PREF_WORKERS) as executor:
    futures = {executor.submit(_sync_one, p, y): (p, y) for p, y in tasks}
    for fut in as_completed(futures):
        pref, year, cnt, exc = fut.result()
        results[(pref, year)] = cnt if exc is None else -1

phase1_elapsed = time.time() - total_start
print(f"\nPhase 1 完了: {phase1_elapsed:.0f}秒 ({phase1_elapsed/60:.1f}分)")

# ── Phase 2: Parquet → DuckDB 一括インポート ────────────────────────────────
print("\n=== Phase 2: DB インポート ===")
imported_total = 0
for pref in TARGET_PREFS:
    for year in TARGET_YEARS:
        p = processed_dir / f"land_prices_y{year}_pref{pref}_pc0.parquet"
        if p.exists():
            df_p = pd.read_parquet(p)
            if not df_p.empty:
                n = db.upsert_land_prices(conn, df_p)
                imported_total += n
                print(f"  ✓ {PREF_NAMES.get(pref, pref)} {year}年: {len(df_p):,} 件 → DB")
        else:
            print(f"  ⚠ {PREF_NAMES.get(pref, pref)} {year}年: Parquetなし（取得失敗）")

print(f"\n合計インポート: {imported_total:,} 件")

total_elapsed = time.time() - total_start
print(f"総所要時間: {total_elapsed:.0f}秒 ({total_elapsed/60:.1f}分)")
print(f"終了: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Streamlit 起動中の場合はキャッシュをクリア
try:
    import streamlit as st
    st.cache_data.clear()
    print("\n✓ Streamlit キャッシュをクリアしました")
except Exception:
    pass

# ── 結果サマリー ─────────────────────────────────────────────────────────────
print("\n=== 取得結果サマリー ===")
for pref in TARGET_PREFS:
    for year in TARGET_YEARS:
        cnt = results.get((pref, year), -1)
        mark = "✓" if cnt > 0 else ("⚠" if cnt == 0 else "✗")
        label = f"{cnt:,} 件" if cnt >= 0 else "エラー"
        print(f"  {mark} {PREF_NAMES.get(pref, pref)} ({pref}) {year}年: {label}")

## 4. DB の状態確認

In [5]:
import db
stats = db.get_stats(conn)
print("DB 総レコード:", stats["total_records"])
print("格納年度:", stats["available_years"])
for year, cnt in stats["year_counts"].items():
    print(f"  {year}年: {cnt:,} 件")

DB 総レコード: 38227
格納年度: [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018]
  2026年: 4,094 件
  2025年: 4,094 件
  2024年: 4,304 件
  2023年: 4,304 件
  2022年: 4,288 件
  2021年: 4,288 件
  2020年: 4,284 件
  2019年: 4,285 件
  2018年: 4,286 件


## 5. Parquet ファイル確認

In [6]:
from pathlib import Path
import pandas as pd

processed_dir = Path("data/processed")
for p in sorted(processed_dir.glob("*.parquet")):
    tmp = pd.read_parquet(p)
    print(f"{p.name}: {len(tmp):,} 行")

land_prices_y2018_pc0.parquet: 1,664 行
land_prices_y2018_pref13_pc0.parquet: 1,594 行
land_prices_y2018_pref47_pc0.parquet: 70 行
land_prices_y2019_pc0.parquet: 1,664 行
land_prices_y2019_pref13_pc0.parquet: 1,594 行
land_prices_y2019_pref47_pc0.parquet: 70 行
land_prices_y2020_pc0.parquet: 1,594 行
land_prices_y2020_pref13_pc0.parquet: 1,594 行
land_prices_y2020_pref47_pc0.parquet: 0 行
land_prices_y2021_pc0.parquet: 1,665 行
land_prices_y2021_pref13_pc0.parquet: 1,595 行
land_prices_y2021_pref47_pc0.parquet: 70 行
land_prices_y2022_pc0.parquet: 1,665 行
land_prices_y2022_pref13_pc0.parquet: 1,595 行
land_prices_y2022_pref47_pc0.parquet: 70 行
land_prices_y2023_pc0.parquet: 1,670 行
land_prices_y2023_pref13_pc0.parquet: 1,600 行
land_prices_y2023_pref47_pc0.parquet: 70 行
land_prices_y2024_pc0.parquet: 1,670 行
land_prices_y2024_pref13_pc0.parquet: 1,600 行
land_prices_y2024_pref47_pc0.parquet: 70 行
land_prices_y2025_pc0.parquet: 1,594 行
land_prices_y2025_pref13_pc0.parquet: 1,594 行
land_prices_y2025_pr

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

processed_dir = Path("data/processed")

PREF_LABEL = {
    "01": "北海道", "13": "東京都", "14": "神奈川県",
    "23": "愛知県", "27": "大阪府", "40": "福岡県", "47": "沖縄県",
}

# ── Parquet 読み込み ─────────────────────────────────────────────
frames = []
print("=== Parquet ファイル確認 ===")
for year in TARGET_YEARS:
    for pref in TARGET_PREFS:
        p = processed_dir / f"land_prices_y{year}_pref{pref}_pc0.parquet"
        if p.exists():
            df_tmp = pd.read_parquet(p)
            if not df_tmp.empty:
                frames.append(df_tmp)
            print(f"  {p.name}: {len(df_tmp):,} 行")
        else:
            print(f"  {year}年/{pref}: ファイルなし")

if not frames:
    print("\n⚠ データなし。先に同期セルを実行してください。")
else:
    df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["point_id", "year"])
    df_all = df_all[df_all["point_id"].notna()]
    if "prefecture_code" not in df_all.columns:
        df_all["prefecture_code"] = df_all["point_id"].astype(str).str[:2]
    df_all["pref_label"] = df_all["prefecture_code"].map(PREF_LABEL).fillna(df_all["prefecture_code"])

    latest_year = int(df_all["year"].max())
    lat_col = "lat" if "lat" in df_all.columns else "latitude"
    lon_col = "lon" if "lon" in df_all.columns else "longitude"
    print(f"\n読み込み完了: {len(df_all):,} 件 / 年度: {sorted(df_all['year'].unique())} / 都道府県: {sorted(df_all['prefecture_code'].unique())}")

    # ── 1. 地点マップ（最新年） ──────────────────────────────────
    df_latest = df_all[df_all["year"] == latest_year].copy()
    fig_map = px.scatter_map(
        df_latest,
        lat=lat_col, lon=lon_col,
        color="pref_label",
        hover_data={"price_yen_per_sqm": ":,.0f", "city_name": True, "pref_label": False},
        title=f"公示地価 地点マップ（{latest_year}年）",
        map_style="carto-positron",
        zoom=4,
        center={"lat": 36.0, "lon": 136.0},
        height=550,
        opacity=0.7,
    )
    fig_map.update_layout(legend_title_text="都道府県")
    fig_map.show()

    # ── 2. 都道府県別 年次平均価格推移 ──────────────────────────
    ts = (
        df_all.groupby(["year", "pref_label"])["price_yen_per_sqm"]
        .mean().reset_index()
        .rename(columns={"price_yen_per_sqm": "avg_price"})
    )
    fig_ts = px.line(
        ts, x="year", y="avg_price", color="pref_label",
        markers=True,
        title="公示地価 都道府県別 年次平均価格推移",
        labels={"year": "年", "avg_price": "平均価格 (円/m²)", "pref_label": "都道府県"},
        height=420,
    )
    fig_ts.update_traces(line_width=2, marker_size=8)
    fig_ts.update_yaxes(tickformat="¥,.0f")
    fig_ts.show()

    # ── 3. 都道府県別 記述統計（最新年） ────────────────────────
    print(f"\n--- 都道府県別 記述統計（{latest_year}年） ---")
    summary = (
        df_latest.groupby("pref_label")["price_yen_per_sqm"]
        .agg(件数="count", 平均="mean", 中央値="median", 最大="max", 最小="min")
        .round(0).astype(int)
    )
    display(summary)